In [2]:
# Imports 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import collections
import os
 
warnings.filterwarnings('ignore')
 
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
 
from sklearn.model_selection import (
    train_test_split, StratifiedKFold,
    cross_val_score, GridSearchCV, RandomizedSearchCV
)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, classification_report, roc_curve
)
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss, cohen_kappa_score, make_scorer
 
# ✅ Use imblearn Pipeline so SMOTE runs inside each CV fold
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
 
import joblib
 
RANDOM_STATE = 42
os.makedirs('outputs', exist_ok=True)
print('All imports successful.')
 

All imports successful.


In [3]:
# Load cleaned CKD dataset
ckd_df = pd.read_csv('../data/chronic_kidney_disease/ckd_cleaned.csv')
 
print('Dataset loaded successfully')
print('Shape:', ckd_df.shape)
print('\nColumn names:')
print(ckd_df.columns.tolist())
print(ckd_df.head())

Dataset loaded successfully
Shape: (400, 25)

Column names:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane', 'classification']
    age    bp     sg   al   su  rbc   pc  pcc   ba    bgr  ...   pcv      wc  \
0  48.0  80.0  1.020  1.0  0.0  0.0  0.0  0.0  0.0  121.0  ...  44.0  7800.0   
1   7.0  50.0  1.020  4.0  0.0  0.0  0.0  0.0  0.0  121.0  ...  38.0  6000.0   
2  62.0  80.0  1.010  2.0  3.0  0.0  0.0  0.0  0.0  423.0  ...  31.0  7500.0   
3  48.0  70.0  1.005  4.0  0.0  0.0  1.0  1.0  0.0  117.0  ...  32.0  6700.0   
4  51.0  80.0  1.010  2.0  0.0  0.0  0.0  0.0  0.0  106.0  ...  35.0  7300.0   

    rc  htn   dm  cad  appet   pe  ane  classification  
0  5.2  1.0  1.0  0.0    0.0  0.0  0.0               1  
1  4.8  0.0  0.0  0.0    0.0  0.0  0.0               1  
2  4.8  0.0  1.0  0.0    1.0  0.0  1.0               1  
3  3.9  1.0  0.0  0.0    1.0  1.0  1.0               

In [4]:
# Quick sanity check — missing values and data types
print('Missing values:')
print(ckd_df.isnull().sum())
print('\nData types:')
print(ckd_df.dtypes)
print('\nTarget distribution (classification):')
print(ckd_df['classification'].value_counts())

Missing values:
age               0
bp                0
sg                0
al                0
su                0
rbc               0
pc                0
pcc               0
ba                0
bgr               0
bu                0
sc                0
sod               0
pot               0
hemo              0
pcv               0
wc                0
rc                0
htn               0
dm                0
cad               0
appet             0
pe                0
ane               0
classification    0
dtype: int64

Data types:
age               float64
bp                float64
sg                float64
al                float64
su                float64
rbc               float64
pc                float64
pcc               float64
ba                float64
bgr               float64
bu                float64
sc                float64
sod               float64
pot               float64
hemo              float64
pcv               float64
wc                float64
rc              

In [5]:
# Separate features (X) and target (y)
X = ckd_df.drop(columns=['classification'])
y = ckd_df['classification']   # 0 = not CKD, 1 = CKD
 
print('Features shape:', X.shape)
print('Target shape:  ', y.shape)
print('\nClass distribution:')
print(y.value_counts())
print(f'\nClass imbalance ratio — Not CKD : CKD = '
      f'{y.value_counts()[0]}:{y.value_counts()[1]}')
 

Features shape: (400, 24)
Target shape:   (400,)

Class distribution:
classification
1    250
0    150
Name: count, dtype: int64

Class imbalance ratio — Not CKD : CKD = 150:250


In [6]:
# Train-test split (stratified to maintain class distribution)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)
 
print(f'Train : {X_train.shape[0]} samples')
print(f'Test  : {X_test.shape[0]} samples')
print('\nTrain class distribution:')
print(y_train.value_counts())
print('\nTest class distribution:')
print(y_test.value_counts())

Train : 320 samples
Test  : 80 samples

Train class distribution:
classification
1    200
0    120
Name: count, dtype: int64

Test class distribution:
classification
1    50
0    30
Name: count, dtype: int64


In [7]:
# Scaling (fit on train only) 
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit + transform
X_test_scaled  = scaler.transform(X_test)         # transform only — no leakage
 
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled  = pd.DataFrame(X_test_scaled,  columns=X.columns)
 
print('Scaling done (fit on train only — no leakage).')
print('\nTrain scaled stats (mean ≈ 0, std ≈ 1):')
print(X_train_scaled.describe().loc[['mean', 'std']].round(3))
 

Scaling done (fit on train only — no leakage).

Train scaled stats (mean ≈ 0, std ≈ 1):
        age     bp     sg     al     su    rbc     pc    pcc     ba    bgr  \
mean -0.000  0.000  0.000  0.000 -0.000  0.000  0.000  0.000  0.000 -0.000   
std   1.002  1.002  1.002  1.002  1.002  1.002  1.002  1.002  1.002  1.002   

      ...   hemo    pcv     wc     rc    htn     dm    cad  appet     pe  \
mean  ... -0.000  0.000  0.000 -0.000  0.000  0.000  0.000 -0.000 -0.000   
std   ...  1.002  1.002  1.002  1.002  1.002  1.002  1.002  1.002  1.002   

        ane  
mean  0.000  
std   1.002  

[2 rows x 24 columns]


In [8]:
# Cell 7: CV Setup 
#SMOTE is NOT applied here — it lives inside each pipeline below
skf          = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
kappa_scorer = make_scorer(cohen_kappa_score)
 
def cv_report(pipeline, X, y, label):
    """
    Honest 5-fold CV: SMOTE runs inside each fold via ImbPipeline.
    No synthetic samples ever leak into the validation fold.
    """
    acc   = cross_val_score(pipeline, X, y, cv=skf, scoring='accuracy')
    roc   = cross_val_score(pipeline, X, y, cv=skf, scoring='roc_auc')
    kappa = cross_val_score(pipeline, X, y, cv=skf, scoring=kappa_scorer)
    print(f'\n[{label}]  5-Fold CV (SMOTE inside fold)')
    print(f'  Accuracy : {acc.mean():.4f}  ±  {acc.std():.4f}')
    print(f'  ROC-AUC  : {roc.mean():.4f}  ±  {roc.std():.4f}')
    print(f'  Kappa    : {kappa.mean():.4f}  ±  {kappa.std():.4f}  ← KEY')
    return {'Accuracy': acc.mean(), 'ROC-AUC': roc.mean(), 'Kappa': kappa.mean()}
 
cv_results = {}
print('CV setup ready.')

CV setup ready.


# Model 1: Logistic Regression

In [9]:
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

RANDOM_STATE = 42
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Step: Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

# Logistic Regression pipeline
lr_pipeline = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('classifier', LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=RANDOM_STATE
    ))
])

# Parameter grid
lr_param_grid = {
    'classifier__C': [0.01, 0.1, 1, 10, 100],
    'classifier__penalty': ['l1', 'l2'],
    'classifier__solver': ['liblinear']
}

# GridSearchCV
lr_grid = GridSearchCV(
    estimator=lr_pipeline,
    param_grid=lr_param_grid,
    cv=skf,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=0
)

# Fit on raw training data
lr_grid.fit(X_train, y_train)

print("Best LR params :", lr_grid.best_params_)
print("Best CV ROC-AUC:", round(lr_grid.best_score_, 4))

lr_best = lr_grid.best_estimator_
cv_results['Logistic Regression'] = cv_report(
    lr_best, X_train, y_train, 'Logistic Regression'
)

X_train: (320, 24)
X_test : (80, 24)
y_train: (320,)
y_test : (80,)
Best LR params : {'classifier__C': 0.01, 'classifier__penalty': 'l2', 'classifier__solver': 'liblinear'}
Best CV ROC-AUC: 1.0

[Logistic Regression]  5-Fold CV (SMOTE inside fold)
  Accuracy : 0.8906  ±  0.0198
  ROC-AUC  : 1.0000  ±  0.0000
  Kappa    : 0.7799  ±  0.0374  ← KEY


In [10]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Predictions
y_pred = lr_best.predict(X_test)
y_proba = lr_best.predict_proba(X_test)[:, 1]

# Metrics
print("\nTest Set Evaluation (Logistic Regression)")
print("ROC-AUC:", round(roc_auc_score(y_test, y_proba), 4))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


Test Set Evaluation (Logistic Regression)
ROC-AUC: 1.0

Classification Report:
               precision    recall  f1-score   support

           0       0.77      1.00      0.87        30
           1       1.00      0.82      0.90        50

    accuracy                           0.89        80
   macro avg       0.88      0.91      0.89        80
weighted avg       0.91      0.89      0.89        80


Confusion Matrix:
 [[30  0]
 [ 9 41]]


# Model 2 - Random Forest

In [11]:
from sklearn.ensemble import RandomForestClassifier

rf_param_grid = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [5, 10, 15],
    'classifier__min_samples_split': [2, 5],
    'classifier__min_samples_leaf': [1, 2],
    'classifier__max_features': ['sqrt', 'log2']
}

rf_pipeline = ImbPipeline([
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('classifier', RandomForestClassifier(
        class_weight='balanced',
        random_state=RANDOM_STATE
    ))
])

rf_grid = GridSearchCV(
    rf_pipeline,
    rf_param_grid,
    cv=skf,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=0
)

# use RAW training data
rf_grid.fit(X_train, y_train)

print('Best RF params :', rf_grid.best_params_)
print('Best CV ROC-AUC:', round(rf_grid.best_score_, 4))

rf_best = rf_grid.best_estimator_

cv_results['Random Forest'] = cv_report(
    rf_best, X_train, y_train, 'Random Forest'
)

Best RF params : {'classifier__max_depth': 5, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 300}
Best CV ROC-AUC: 0.9998

[Random Forest]  5-Fold CV (SMOTE inside fold)
  Accuracy : 0.9938  ±  0.0077
  ROC-AUC  : 0.9998  ±  0.0004
  Kappa    : 0.9866  ±  0.0165  ← KEY


In [12]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

y_pred_rf = rf_best.predict(X_test)
y_proba_rf = rf_best.predict_proba(X_test)[:, 1]

print("\nTest Set Evaluation (Random Forest)")
print("ROC-AUC:", round(roc_auc_score(y_test, y_proba_rf), 4))
print("\nClassification Report:\n", classification_report(y_test, y_pred_rf))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))


Test Set Evaluation (Random Forest)
ROC-AUC: 1.0

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        30
           1       1.00      1.00      1.00        50

    accuracy                           1.00        80
   macro avg       1.00      1.00      1.00        80
weighted avg       1.00      1.00      1.00        80


Confusion Matrix:
 [[30  0]
 [ 0 50]]


# Model 3 - XGBoost

In [13]:
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV

xgb_param_grid = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [3, 5, 7],
    'classifier__learning_rate': [0.01, 0.05, 0.1],
    'classifier__subsample': [0.7, 0.8, 1.0],
    'classifier__colsample_bytree': [0.7, 0.8, 1.0],
    'classifier__reg_alpha': [0, 0.1, 0.5],
    'classifier__reg_lambda': [1, 1.5, 2]
}

xgb_pipeline = ImbPipeline([
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('classifier', XGBClassifier(
        eval_metric='logloss',
        random_state=RANDOM_STATE
    ))
])

xgb_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=xgb_param_grid,
    n_iter=30,
    cv=skf,
    scoring='roc_auc',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=0
)

# use RAW training data
xgb_search.fit(X_train, y_train)

print('Best XGB params :', xgb_search.best_params_)
print('Best CV ROC-AUC :', round(xgb_search.best_score_, 4))

xgb_best = xgb_search.best_estimator_

cv_results['XGBoost'] = cv_report(
    xgb_best, X_train, y_train, 'XGBoost'
)

Best XGB params : {'classifier__subsample': 1.0, 'classifier__reg_lambda': 1, 'classifier__reg_alpha': 0.5, 'classifier__n_estimators': 100, 'classifier__max_depth': 5, 'classifier__learning_rate': 0.1, 'classifier__colsample_bytree': 0.8}
Best CV ROC-AUC : 0.9988

[XGBoost]  5-Fold CV (SMOTE inside fold)
  Accuracy : 0.9844  ±  0.0242
  ROC-AUC  : 0.9988  ±  0.0017
  Kappa    : 0.9666  ±  0.0516  ← KEY


In [14]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

y_pred_xgb = xgb_best.predict(X_test)
y_proba_xgb = xgb_best.predict_proba(X_test)[:, 1]

print("\nTest Set Evaluation (XGBoost)")
print("ROC-AUC:", round(roc_auc_score(y_test, y_proba_xgb), 4))
print("\nClassification Report:\n", classification_report(y_test, y_pred_xgb))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_xgb))


Test Set Evaluation (XGBoost)
ROC-AUC: 1.0

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        30
           1       1.00      1.00      1.00        50

    accuracy                           1.00        80
   macro avg       1.00      1.00      1.00        80
weighted avg       1.00      1.00      1.00        80


Confusion Matrix:
 [[30  0]
 [ 0 50]]


In [18]:
# HOLD-OUT TEST SET EVALUATION

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
    cohen_kappa_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    roc_curve
)
from sklearn.calibration import calibration_curve
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

# create output folder
os.makedirs("outputs", exist_ok=True)

print('HOLD-OUT TEST SET RESULTS')

models = {
    'Logistic Regression': lr_best,
    'Random Forest'      : rf_best,
    'XGBoost'            : xgb_best,
}

test_results = {}

for name, model in models.items():
    # IMPORTANT: use RAW X_test, not X_test_scaled
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    acc       = accuracy_score(y_test, y_pred)
    roc       = roc_auc_score(y_test, y_prob)
    f1        = f1_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall    = recall_score(y_test, y_pred)
    kappa     = cohen_kappa_score(y_test, y_pred)
    brier     = brier_score_loss(y_test, y_prob)

    test_results[name] = {
        'Accuracy' : acc,
        'ROC-AUC'  : roc,
        'F1'       : f1,
        'Precision': precision,
        'Recall'   : recall,
        'Kappa'    : kappa,
        'Brier'    : brier,
    }

    print(f'\n── {name} ──')
    print(f'  Accuracy  : {acc:.4f}')
    print(f'  ROC-AUC   : {roc:.4f}')
    print(f'  F1        : {f1:.4f}')
    print(f'  Precision : {precision:.4f}')
    print(f'  Recall    : {recall:.4f}')
    print(f'  Kappa     : {kappa:.4f}')
    print(f'  Brier     : {brier:.4f}  (lower = better calibrated)')
    print(classification_report(y_test, y_pred, target_names=['Not CKD', 'CKD']))


HOLD-OUT TEST SET RESULTS

── Logistic Regression ──
  Accuracy  : 0.8875
  ROC-AUC   : 1.0000
  F1        : 0.9011
  Precision : 1.0000
  Recall    : 0.8200
  Kappa     : 0.7736
  Brier     : 0.0754  (lower = better calibrated)
              precision    recall  f1-score   support

     Not CKD       0.77      1.00      0.87        30
         CKD       1.00      0.82      0.90        50

    accuracy                           0.89        80
   macro avg       0.88      0.91      0.89        80
weighted avg       0.91      0.89      0.89        80


── Random Forest ──
  Accuracy  : 1.0000
  ROC-AUC   : 1.0000
  F1        : 1.0000
  Precision : 1.0000
  Recall    : 1.0000
  Kappa     : 1.0000
  Brier     : 0.0103  (lower = better calibrated)
              precision    recall  f1-score   support

     Not CKD       1.00      1.00      1.00        30
         CKD       1.00      1.00      1.00        50

    accuracy                           1.00        80
   macro avg       1.00      

In [ ]:
# OVERFITTING CHECK — CV vs TEST GAP

print('\n' + '='*60)
print('OVERFITTING CHECK — CV vs Hold-out Gap')
print('='*60)
print(f'{"Model":<22} {"CV AUC":>8}  {"Test AUC":>9}  {"Gap":>8}')
print('-'*55)

for name in models:
    cv_auc   = cv_results[name]['ROC-AUC']
    test_auc = test_results[name]['ROC-AUC']
    gap      = cv_auc - test_auc
    flag     = 'Overfit' if gap > 0.05 else 'OK'
    print(f'{name:<22} {cv_auc:>8.4f}  {test_auc:>9.4f}  {gap:>+8.4f}  {flag}')


OVERFITTING CHECK — CV vs Hold-out Gap
Model                    CV AUC   Test AUC       Gap
-------------------------------------------------------
Logistic Regression      1.0000     1.0000   +0.0000  OK
Random Forest            0.9998     1.0000   -0.0002  OK
XGBoost                  0.9988     1.0000   -0.0012  OK
